### **1. Do elastic products generate more revenue or inelastic ones?**

In [0]:
%sql
WITH revenue AS (
    SELECT
        Description,
        SUM(Quantity * Price) AS total_revenue
    FROM retail
    GROUP BY Description
)

SELECT
    e.elasticity,
    r.total_revenue
FROM elasticity_table e
JOIN revenue r
ON e.Description = r.Description

elasticity,total_revenue
0.118,3504.840000000003
0.87,636.75
-0.241,49539.69000000054
-0.917,1169.5500000000015
-0.336,18551.699999999928
-0.664,6259.490000000017
0.525,170.86999999999986
-0.478,13075.14999999994
0.681,25065.069999999992
-3.7,3657.0


Databricks visualization. Run in Databricks to view.

###Conclusion:
 Products with near-zero elasticity generate the highest revenue, while highly elastic and positive-elastic products contribute significantly less. Stable demand (low sensitivity) drives revenue, not reactive demand. Hence, Revenue peaks where demand is least sensitive to price.

### **2. Statistics of Elasticity (category wise) vs revenue**


In [0]:
%sql
WITH revenue AS (
    SELECT
        Description,
        SUM(Quantity * Price) AS total_revenue
    FROM retail
    GROUP BY Description
)

SELECT
    e.category,
    COUNT(*) AS product_count,
    AVG(r.total_revenue) AS avg_revenue,
    SUM(r.total_revenue) AS total_revenue,
    STDDEV(r.total_revenue) AS stddev_revenue,
    MIN(r.total_revenue) AS min_revenue,
    MAX(r.total_revenue) AS max_revenue,
    (MAX(r.total_revenue) - MIN(r.total_revenue)) AS range_revenue
FROM elasticity_table e
JOIN revenue r
ON e.Description = r.Description
GROUP BY e.category

category,product_count,avg_revenue,total_revenue,stddev_revenue,min_revenue,max_revenue,range_revenue
anomalous_positive,384,7257.853229166682,2787015.6400000057,14393.469260387561,15.44,150935.55999999988,150920.11999999988
inelastic,3522,4797.584074673492,1.689709111100004E7,13709.835896240891,3.69,344563.2500000036,344559.5600000036
elastic,370,2227.0850540540546,824021.4700000002,5313.817004986651,5.859999999999999,66964.99000000028,66959.13000000028


Databricks visualization. Run in Databricks to view.

### The following conclusions are made:-
1. The product count of Anomalous Positive goods are just 384 but the avg revenue pulled is 7258, that is because the average here is biased and affected by fewer high values. So to know it better, I checked standard deviations of each category, which came high at 14,393 for this category.

2. The inelastic goods stand with 3522 products, standard deviation around 13.7k which is lesser than anomalous positive category. This large category contributed to 16.9 Million revenue generation.

3. Finally the elastic goods which follows law of demand normally, stands at 370 product count, a standard deviation of around 5.3k which is way too lesser than above two category. However despite looking stable at first, the total revenue sum is just 0.82 Million, which ranks this category as 3rd to contribute in revenue.

### **3. Elasticity vs Sales Volume**

In [0]:
%sql
WITH volume AS (
    SELECT
        Description,
        SUM(Quantity) AS total_qty,
        MIN(Quantity) AS min_qty,
        MAX(Quantity) AS max_qty,
        percentile_approx(Quantity, 0.5) AS median_qty
    FROM retail
    GROUP BY Description
)

SELECT
    e.category,
    AVG(v.total_qty) AS avg_total_qty,
    AVG(v.min_qty) AS avg_min_qty,
    AVG(v.max_qty) AS avg_max_qty,
    AVG(v.median_qty) AS avg_median_qty
FROM elasticity_table e
JOIN volume v
ON e.Description = v.Description
GROUP BY e.category

category,avg_total_qty,avg_min_qty,avg_max_qty,avg_median_qty
anomalous_positive,3621.1015625,1.3177083333333333,188.6484375,5.427083333333333
elastic,1609.3297297297297,2.6945945945945944,197.03513513513514,7.883783783783784
inelastic,2611.7603634298694,1.0826235093696763,302.0082339579784,4.3483816013628624


The Anomalous positive goods are more in demand then any others. The validation is done by looking at other statistics that is similar accross all 3 categories, so their is no bias at all.

### 

### **4. Elasticity vs Price Level**

In [0]:
%sql
WITH price_stats AS (
    SELECT
        Description,
        AVG(Price) AS avg_price,
        MIN(Price) AS min_price,
        MAX(Price) AS max_price,
        percentile_approx(Price, 0.5) AS median_price
    FROM retail
    GROUP BY Description
)

SELECT
    e.category,
    AVG(p.avg_price) AS avg_price,
    AVG(p.min_price) AS avg_min_price,
    AVG(p.max_price) AS avg_max_price,
    AVG(p.median_price) AS avg_median_price
FROM elasticity_table e
JOIN price_stats p
ON e.Description = p.Description
GROUP BY e.category

category,avg_price,avg_min_price,avg_max_price,avg_median_price
anomalous_positive,26.199976480558533,3.3724739583333334,42.78151041666666,21.03216145833333
elastic,6.742479619548906,4.702864864864864,9.972216216216218,6.60745945945946
inelastic,4.068468813855145,2.148100795002842,18.50818285065304,3.5169363997728538


The price of anomalous positive goods have a strong bias to pull average, the average max price is **42.8 units** that is the reason of such high revenue.

The average max price in the elastic category is **9.97 units** with a way less than the anomalous positive goods having avg. max price **42.8**, that made very less revenue totally in millions. This shows that *few anomalous products perform extremely well NOT all anomalous products*.

The inelastic goods stand with avg max price of **18.5** which made them more normal than the other two because the demand is more than **3.5k** for them whereas it was **21** for anomalous positive goods in median price terms, which still have biased prices. The inelastic goods made **16.9 million revenue** which made them **1st position** among all.

---

### ***Important conclusion made***

The inelastic goods must be pursued in sales maximization strategy, however the anomalous positive goods must be pursued to maximize sales but only those which made the jump in average, means goods with high demand and high price.

---

### **The risk analysis is as follows:**

* **Some anomalous goods:** High Risk, High Revenue
* **Inelastic goods overall:** Low Risk, High Revenue
* **Elastic goods overall:** Low Risk, Low Revenue
